In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Link Prediction with Logistic Regression and Enhanced Topological Features
=========================================================================
This script builds an undirected graph from an XML file (e.g., oncogene_pathways.xml),
computes a set of topological features (CN, Jaccard, Adamic‑Adar, Preferential Attachment,
Resource Allocation, Katz, LHN, PageRank similarity) for existing and sampled non‑edges,
trains a logistic regression model, predicts missing edges, and validates predictions
against the STRING database.

Usage:
    python 06_link_prediction_logistic_regression.py
"""

import os
import random
import time
import xml.etree.ElementTree as ET
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, classification_report

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)

# ----------------------------------------------------------------------
# 1. Build graph from XML
# ----------------------------------------------------------------------
def clean_gene_name(name):
    """Clean node name by removing mutation markers and special suffixes."""
    name = name.rstrip('*')
    if '+' in name:
        name = name.split('+')[0]
    name = name.replace('(cyto)', '').replace('(mito)', '').replace('(outer)', '').replace('(inner)', '').strip()
    return name

def build_graph_from_xml(xml_path):
    """Parse XML and return a directed graph."""
    tree = ET.parse(xml_path)
    root = tree.getroot()
    G = nx.DiGraph()
    for pathway in root.findall('pathway'):
        node_list = pathway.find('nodeList')
        if node_list is not None:
            for node in node_list.findall('node'):
                node_name = node.get('name')
                if node_name:
                    cleaned = clean_gene_name(node_name)
                    G.add_node(node_name, cleaned=cleaned)
        edge_list = pathway.find('edgeList')
        if edge_list is not None:
            for edge in edge_list.findall('edge'):
                start = edge.find('startNode')
                end = edge.find('endNode')
                if start is not None and end is not None:
                    start_node = start.get('node')
                    end_node = end.get('node')
                    if start_node and end_node:
                        G.add_edge(start_node, end_node)
    print(f"Graph built: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
    return G

# ----------------------------------------------------------------------
# 2. Negative sampling with degree‑bucket matching
# ----------------------------------------------------------------------
def degree_bucket(d, bins):
    if d <= bins[0]:
        return 0
    elif d <= bins[1]:
        return 1
    elif d <= bins[2]:
        return 2
    else:
        return 3

def sample_negative_edges(G, existing_edges, bins, bucket_to_nodes, pos_bucket_dist, nodes):
    """Sample negative edges matching degree distribution of positive edges."""
    neg_edges = []
    for (bu, bv), count in pos_bucket_dist.items():
        candidates_u = bucket_to_nodes[bu]
        candidates_v = bucket_to_nodes[bv]
        sampled = 0
        attempts = 0
        while sampled < count and attempts < count * 20:
            u = random.choice(candidates_u)
            v = random.choice(candidates_v)
            if u != v and (u, v) not in existing_edges and (v, u) not in existing_edges:
                neg_edges.append((u, v))
                sampled += 1
            attempts += 1
        # If still not enough, fill with random nodes
        while sampled < count:
            u = random.choice(nodes)
            v = random.choice(nodes)
            if u != v and (u, v) not in existing_edges and (v, u) not in existing_edges:
                neg_edges.append((u, v))
                sampled += 1
    return neg_edges

# ----------------------------------------------------------------------
# 3. Enhanced topological features
# ----------------------------------------------------------------------
def common_neighbors(G, u, v):
    return len(set(G.neighbors(u)) & set(G.neighbors(v)))

def jaccard_coefficient(G, u, v):
    nu = set(G.neighbors(u))
    nv = set(G.neighbors(v))
    inter = len(nu & nv)
    union = len(nu | nv)
    return inter / union if union > 0 else 0

def adamic_adar(G, u, v):
    common = set(G.neighbors(u)) & set(G.neighbors(v))
    score = 0.0
    for w in common:
        deg = G.degree(w)
        if deg > 1:
            score += 1.0 / np.log(deg)
    return score

def preferential_attachment(G, u, v):
    return G.degree(u) * G.degree(v)

def resource_allocation(G, u, v):
    common = set(G.neighbors(u)) & set(G.neighbors(v))
    return sum(1.0 / G.degree(w) for w in common if G.degree(w) > 0)

def katz_index(G, u, v, beta=0.01):
    """Katz index approximation (paths up to length 3)."""
    score = 0
    if G.has_edge(u, v):
        score += beta
    common = set(G.neighbors(u)) & set(G.neighbors(v))
    score += beta**2 * len(common)
    # length‑3 paths
    for w in G.neighbors(u):
        for x in G.neighbors(w):
            if G.has_edge(x, v) and x != u and w != v:
                score += beta**3
    return score

def lhn_similarity(G, u, v):
    """Leicht‑Holme‑Newman similarity."""
    common = set(G.neighbors(u)) & set(G.neighbors(v))
    deg_u = G.degree(u)
    deg_v = G.degree(v)
    if deg_u == 0 or deg_v == 0:
        return 0
    return sum(1.0 / G.degree(w) for w in common) / (deg_u * deg_v)

def pagerank_similarity(pr, u, v):
    return pr[u] * pr[v]

def compute_features(G, edges, pr):
    """Compute all features for a list of edges."""
    features = []
    for u, v in edges:
        cn = common_neighbors(G, u, v)
        jc = jaccard_coefficient(G, u, v)
        aa = adamic_adar(G, u, v)
        pa = preferential_attachment(G, u, v)
        ra = resource_allocation(G, u, v)
        katz = katz_index(G, u, v)
        lhn = lhn_similarity(G, u, v)
        pr_sim = pagerank_similarity(pr, u, v)
        features.append([cn, jc, aa, pa, ra, katz, lhn, pr_sim])
    return np.array(features)

# ----------------------------------------------------------------------
# 4. STRING validation helper (with retries)
# ----------------------------------------------------------------------
def create_session_with_retries():
    session = requests.Session()
    retries = Retry(total=3, backoff_factor=1, status_forcelist=[500, 502, 503, 504])
    session.mount('https://', HTTPAdapter(max_retries=retries))
    return session

session = create_session_with_retries()

def check_string_interaction(protein1, protein2, species=9606, max_retries=2):
    """Query STRING database to check if an interaction is known."""
    url = f"https://string-db.org/api/json/network?identifiers={protein1}%0D{protein2}&species={species}"
    for attempt in range(max_retries):
        try:
            resp = session.get(url, timeout=30)
            if resp.status_code == 200:
                data = resp.json()
                for item in data:
                    if (item['preferredName_A'] == protein1 and item['preferredName_B'] == protein2) or \
                       (item['preferredName_A'] == protein2 and item['preferredName_B'] == protein1):
                        score = float(item.get('score', 0)) / 1000.0
                        return True, score
                return False, 0.0
            else:
                time.sleep(2)
        except Exception as e:
            print(f"Attempt {attempt+1}/{max_retries} failed: {e}")
            time.sleep(3)
    return None, 0.0

# ----------------------------------------------------------------------
# 5. Main pipeline
# ----------------------------------------------------------------------
def main():
    # Input file
    xml_file = "oncogene_pathways.xml"
    if not os.path.exists(xml_file):
        raise FileNotFoundError(f"Input file {xml_file} not found. Please run previous steps first.")

    # Build graph
    G_dir = build_graph_from_xml(xml_file)
    G_und = G_dir.to_undirected()
    print(f"Original directed graph: {G_dir.number_of_nodes()} nodes, {G_dir.number_of_edges()} edges")
    print(f"Undirected graph: {G_und.number_of_nodes()} nodes, {G_und.number_of_edges()} edges")

    existing_edges = set(G_und.edges())
    nodes = list(G_und.nodes())
    print(f"Positive samples (existing edges): {len(existing_edges)}")

    # Compute degree buckets
    degree = dict(G_und.degree())
    degree_vals = list(degree.values())
    bins = np.percentile(degree_vals, [25, 50, 75])

    # Bucket distribution of positive edges
    pos_bucket_dist = defaultdict(int)
    for u, v in existing_edges:
        bu = degree_bucket(degree[u], bins)
        bv = degree_bucket(degree[v], bins)
        key = (bu, bv) if bu <= bv else (bv, bu)
        pos_bucket_dist[key] += 1

    # Nodes per bucket
    bucket_to_nodes = defaultdict(list)
    for n, d in degree.items():
        bucket_to_nodes[degree_bucket(d, bins)].append(n)

    # Sample negatives
    neg_edges = sample_negative_edges(G_und, existing_edges, bins, bucket_to_nodes, pos_bucket_dist, nodes)
    print(f"Negative samples: {len(neg_edges)}")

    # Combine positive and negative samples
    all_edges = list(existing_edges) + neg_edges
    y = [1] * len(existing_edges) + [0] * len(neg_edges)

    # Precompute PageRank
    pr = nx.pagerank(G_und, alpha=0.85)

    # Compute features
    print("Computing enhanced features...")
    X = compute_features(G_und, all_edges, pr)
    feature_names = ['CN', 'Jaccard', 'AA', 'PA', 'RA', 'Katz', 'LHN', 'PR_sim']
    print(f"Feature matrix shape: {X.shape}")

    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    print(f"Training set: {len(y_train)} (pos {sum(y_train)}, neg {len(y_train)-sum(y_train)})")
    print(f"Test set: {len(y_test)} (pos {sum(y_test)}, neg {len(y_test)-sum(y_test)})")

    # Logistic regression with hyperparameter tuning
    param_grid = {'C': [0.01, 0.1, 1, 10, 100]}
    lr = LogisticRegression(class_weight='balanced', max_iter=1000, solver='lbfgs')
    grid = GridSearchCV(lr, param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_
    print(f"Best parameters: {grid.best_params_}, CV AUC: {grid.best_score_:.4f}")

    # Evaluate on test set
    y_pred_prob = best_model.predict_proba(X_test)[:, 1]
    y_pred = (y_pred_prob > 0.5).astype(int)
    auc = roc_auc_score(y_test, y_pred_prob)
    auprc = average_precision_score(y_test, y_pred_prob)
    acc = accuracy_score(y_test, y_pred)

    print("\n=== Model Evaluation ===")
    print(f"AUC: {auc:.4f}")
    print(f"AUPRC: {auprc:.4f}")
    print(f"Accuracy: {acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

    # Feature importance (coefficients)
    coef = best_model.coef_[0]
    print("\nFeature importance (coefficients):")
    for name, c in zip(feature_names, coef):
        print(f"{name}: {c:.4f}")

    # Plot feature importance
    plt.figure(figsize=(8, 5))
    plt.barh(feature_names, coef)
    plt.xlabel('Coefficient')
    plt.title('Feature Importance in Link Prediction')
    plt.tight_layout()
    plt.savefig('link_pred_feature_importance.png')
    plt.show()

    # Predict all missing edges
    print("\nGenerating all non‑edge pairs...")
    all_pairs = []
    for i in range(len(nodes)):
        u = nodes[i]
        for v in nodes[i+1:]:
            if (u, v) not in existing_edges:
                all_pairs.append((u, v))
    print(f"Total non‑edge pairs: {len(all_pairs)}")

    if all_pairs:
        print("Computing features for all pairs...")
        X_all = compute_features(G_und, all_pairs, pr)
        probs = best_model.predict_proba(X_all)[:, 1]

        pred_df = pd.DataFrame({
            'Node1': [p[0] for p in all_pairs],
            'Node2': [p[1] for p in all_pairs],
            'Link_Probability': probs
        })
        pred_df = pred_df.sort_values('Link_Probability', ascending=False)
        top100 = pred_df.head(100)
        top100.to_csv('top100_predicted_links.csv', index=False)
        print("Top 100 predictions saved to top100_predicted_links.csv")
        print("\nTop 10 predicted links:")
        print(top100.head(10))

        # STRING validation on top 100
        print("\nValidating top 100 predictions with STRING...")
        validation = []
        for idx, row in top100.iterrows():
            p1, p2 = row['Node1'], row['Node2']
            prob = row['Link_Probability']
            known, score = check_string_interaction(p1, p2)
            validation.append({
                'Node1': p1,
                'Node2': p2,
                'Link_Probability': prob,
                'STRING_Known': known,
                'STRING_Score': score
            })
            if (idx + 1) % 20 == 0:
                print(f"Processed {idx+1}/100")
            time.sleep(0.5)

        valid_df = pd.DataFrame(validation)
        valid_df.to_csv('top100_validated_with_string.csv', index=False)
        print("Validation results saved to top100_validated_with_string.csv")

        # Compute precision@k
        k_list = [10, 20, 50, 100]
        print("\n=== STRING Validation Precision ===")
        precisions = []
        for k in k_list:
            sub = valid_df.head(k)
            known_count = sub[sub['STRING_Known'] == True].shape[0]
            precision = known_count / k
            precisions.append(precision)
            print(f"Precision@{k} = {precision:.4f} ({known_count}/{k})")

        # Plot precision@k
        plt.figure(figsize=(6, 4))
        plt.plot(k_list, precisions, marker='o', linestyle='-')
        plt.xlabel('k')
        plt.ylabel('Precision')
        plt.title('Precision@k on STRING Database')
        plt.grid(True)
        plt.savefig('precision_at_k.png')
        plt.show()
    else:
        print("No non‑edge pairs found.")

    print("\nLink prediction and validation completed.")

if __name__ == '__main__':
    main()